# Week 2 · Prompting and Structured Output

**GDG AI Track · Cycle 1: Hands-on Start**

Time: a 2-hour session, plus take-home work (Project 1, Component B).

## By the end of this notebook you can

- Write a prompt that gives the model clear instructions
- Put examples inside a prompt to show the model what you want
- Ask the model for JSON and turn its reply into Python data
- **Validate** the reply (check it), because the model will sometimes break the format
- Run an extractor on 10 messy campus notices and count how many came back valid

## Read this first

- **You do not need to understand every line of code.** When a cell says **Run**, run it and look at the output.
- You only *write* code in cells marked **Your turn**. Each one is short and has a check that says PASS or FAIL.
- Run cells **from top to bottom** with `Shift + Enter`. Don't skip cells.
- If something is badly broken: restart the kernel, then run all cells again (VS Code: **Restart**, then **Run All**).
- The model's answers change from run to run. Yours will not match the examples word for word. That is normal, and it is the reason for Week 3.

| Label | Meaning |
|---|---|
| Read | Read and think |
| Run | Run the cell and look at the output |
| Your turn | Edit the code, then run the cell. It tells you PASS or FAIL |
| Warning | This is where people get stuck |

## Agenda (matches the session slides)

| Part | Topic | Minutes |
|---|---|---|
| (slides) | Demo of Component A, blockers, recap | 10 |
| 1 to 5 | Concept: prompts, examples, JSON, validating | 35 |
| 6 | Lab: messy text to JSON (in pairs) | 45 |
| 7 | Launch the take-home: Component B | 20 |
| | Buffer | 10 |

Part 0 is a two-minute check. Do it **before** class.

# Part 0 · Quick start (before class, 2 minutes)

You set everything up in Week 1. This week needs **no new packages**. You only need to open the same project and check it still works.

1. Put this notebook in your `gdg-ai-track` folder, next to your `.env` file.
2. Open VS Code, then **File → Open Folder** and choose `gdg-ai-track`.
3. Open a terminal (**Terminal → New Terminal**) and check that `(.venv)` shows at the start of the line. If it doesn't, run the activate line again:
   - Mac / Linux: `source .venv/bin/activate`
   - Windows PowerShell: `.venv\Scripts\Activate.ps1`
4. Open this notebook, click **Select Kernel** (top right), choose **Jupyter Kernel**, then **GDG AI Track (Python 3)**.
5. Run the next cell.

> Warning: If you missed Week 1 or something below says PROBLEM, follow `SETUP.md` from Week 1 first. Do it at home, with internet, **before** class.

In [1]:
import os, sys, socket

def internet_ok(host="generativelanguage.googleapis.com", port=443, timeout=3):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

v = sys.version_info
print("OK      " if v >= (3, 11) else "PROBLEM ", f"Python {v.major}.{v.minor} (you need 3.11 or newer)")

try:
    from dotenv import load_dotenv, find_dotenv
    import google.genai
    load_dotenv(find_dotenv(usecwd=True))
    print("OK      tools are installed")
except ImportError:
    print("PROBLEM tools are missing. You are probably on the wrong kernel. Select GDG AI Track (Python 3).")

print("OK      " if os.environ.get("GEMINI_API_KEY") else "PROBLEM ", "API key found in .env")
print("OK      " if internet_ok() else "WARNING ", "internet reaches the Gemini API (the live cells need it)")

OK       Python 3.11 (you need 3.11 or newer)
OK      tools are installed
OK       API key found in .env
OK       internet reaches the Gemini API (the live cells need it)


# Part 1 · Connect and warm up (5 min)

## Run · Connect to the model

This cell reads your key from `.env` and connects. It needs internet.

In [2]:
import os, json, re, time, getpass
from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types

load_dotenv(find_dotenv(usecwd=True))
API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass.getpass("Could/n't find GEMINI_API_KEY in .env. Paste your key (hidden): ")

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.5-flash-lite"
print("Ready. Model:", MODEL)

Ready. Model: gemini-3.5-flash-lite


> Warning: **Model names change.** If you get a `404` or "model not found" error later, change `MODEL` above to a name that works. Your track lead will tell you which one. (Week 1 showed how to list the models.)

## Run · Your Week 1 helpers

These are the helpers from last week, in one cell so that everyone starts equal. `ask` sends a prompt and returns the answer and the token counts. `safe_ask` is the Week 1 take-home: it retries and **never crashes**. One new option this week: `json_mode`. We will use it in Part 4.

You can ignore how they work. Run the cell, then the test under it.

In [3]:
def ask(prompt, system=None, temperature=None, json_mode=False):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if json_mode else None,
    )
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.text or "", response.usage_metadata


def safe_ask(prompt, system=None, temperature=None, json_mode=False):
    for attempt in range(3):
        try:
            text, usage = ask(prompt, system, temperature, json_mode)
            return {
                "prompt": prompt,
                "response": text,
                "model": MODEL,
                "prompt_tokens": usage.prompt_token_count,
                "output_tokens": usage.candidates_token_count,
            }
        except Exception as e:
            if attempt == 2:
                return {"prompt": prompt, "error": str(e)}
            time.sleep(2 ** (attempt + 1))


RESULTS = {}

def check(label, passed):
    RESULTS[label] = bool(passed)
    print("PASS" if passed else "FAIL", "-", label)

def scoreboard():
    print("Passed", sum(RESULTS.values()), "of", len(RESULTS))
    for label, ok in RESULTS.items():
        if not ok:
            print("  still FAIL:", label)

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps(data, indent=2, ensure_ascii=False))

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.loads(f.read())

In [4]:
result = safe_ask("Reply with exactly one word: Ready")
print(result.get("response", result.get("error")))

Ready


If you see an error here instead of a word: read the message. A bad key means a problem with `.env`. A network error means no internet. Fix it before you go on.

---

# Part 2 · A prompt is a set of instructions (10 min)

## Read · The model only knows what you tell it

When you ask a person "what's the date?", they know the context. The model does not. It sees **only the words in your prompt**. If a detail is missing, it **guesses**, and the guess sounds confident.

So a good prompt is closer to a short work order than a question. It says who the model is acting as, what to do, what it is given, what the answer must look like, and what to do when something is unclear.

## Run · Three prompts for the same job

Here is a messy campus notice (made up). Read it, then run the cell. It tries three prompts on the same notice.

In [5]:
POSTED_ON = "2026-11-22"   # the day these notices were posted. We use it to work out "tomorrow", "next Friday", etc.

notice_1 = """HEY GUYS!!! GDG AI workshop this Thursday 26th Nov @ Hall B, 4pm sharp. Bring laptops.
Register at the link in bio. No registration = no entry! pls share"""

prompt_vague = "Tell me about this notice:\n\n" + notice_1
prompt_task = "Extract the title, date, time and venue from this notice:\n\n" + notice_1
prompt_clear = f"""You extract details from campus notices.
Today's date is {POSTED_ON}.

Reply with ONE JSON object and nothing else. Use exactly these keys:
title, date (as YYYY-MM-DD), time (24-hour HH:MM), venue.

Notice:
{notice_1}"""

for name, p in [("VAGUE", prompt_vague), ("TASK", prompt_task), ("CLEAR", prompt_clear)]:
    result = safe_ask(p, temperature=0)
    print("=====", name)
    print(result.get("response", "ERROR: " + result.get("error", "")).strip())
    print()
    time.sleep(1)

===== VAGUE
This is an informal, urgent announcement for a university or community tech workshop. Here is a breakdown of what the notice tells you and the key details you need to know:

### **Key Details:**
*   **What:** Google Developer Group (GDG) AI Workshop. (This means it will likely focus on Artificial Intelligence tools, machine learning, or related tech, hosted by a student or local chapter of Google Developer Groups).
*   **When:** This Thursday, November 26th at **4:00 PM sharp**. (Note: "Sharp" means they likely plan to start on time, and latecomers might be locked out or miss important setup steps).
*   **Where:** Hall B.
*   **What to Bring:** Your laptop (mandatory, as it's a hands-on workshop).
*   **Action Required:** You must register via the link in their social media bio. 
*   **Strict Rule:** **"No registration = no entry!"** (They likely have a strict capacity limit for Hall B).

### **Tone and Vibe:**
*   **Informal & Enthusiastic:** Written in all-caps excitement

Look at the three answers. Which one could a **program** read without a human fixing it?

- The vague prompt gives you an essay.
- The task prompt gives you the right facts, but in a shape that changes every time.
- The clear prompt gives you something a program can use.

## Read · Five parts of a good prompt

| Part | What it does | Example |
|---|---|---|
| Role | Sets the job | "You extract details from campus notices." |
| Context | Facts the model can't know | "Today's date is 2026-11-22." |
| Task | What to do | "Reply with ONE JSON object." |
| Format | What the answer looks like | "Keys: title, date (YYYY-MM-DD), time (HH:MM)" |
| Rules | What to do when unclear | "If a detail is missing, use null. Never guess." |

The **rules** part is the one beginners forget, and it is where most hallucinations are prevented.

## Your turn · Exercise A: `build_prompt`

You will build the prompt function that your extractor uses all week. It takes a notice and returns a full prompt.

The skeleton already has the role, the context, the task and the format. Two `TODO`s are left:

1. Write a **rule** for what the model must do when a detail is missing.
2. Put the **notice** at the end (use `{notice}` inside the string).

Then run the cell. The checks tell you PASS or FAIL.

<details><summary>Stuck? Click for the full answer</summary>

Replace the first `TODO` line with:

```
Rules: If a detail is missing from the notice, use null. Never guess or invent a detail.
```

and the last `TODO` with `{notice}`.

</details>

In [6]:
POSTED_ON = "2026-11-22"

FIELDS = ["title", "type", "date", "time", "venue", "registration_required"]
TYPES = ["event", "deadline", "announcement"]

def build_prompt(notice):
    rule = "Rules: If a detail is missing from the notice, use null. Never guess or invent a detail."
    return f"""You extract details from campus notices.
Today's date is {POSTED_ON}. Use it for dates like "tomorrow" or "next Friday".

Reply with ONE JSON object and nothing else, with exactly these keys:
- title: short name of the thing (text)
- type: "event", "deadline" or "announcement"
- date: YYYY-MM-DD, or null
- time: 24-hour HH:MM, or null
- venue: text, or null
- registration_required: true or false

{rule}

Notice:
{notice}"""

# --- check (don't edit below) ---
p = build_prompt("Choir practice on Friday at 6pm in the Chapel.")
check("build_prompt: no TODO left", "TODO" not in p)
check("build_prompt: the notice is in the prompt", "Choir practice on Friday at 6pm in the Chapel." in p)
check("build_prompt: the notice comes last", p.strip().endswith("in the Chapel."))
check("build_prompt: has a rule about missing details", re.search(r"missing|guess|invent|not (given|mentioned|stated)", p, re.I) is not None)
check("build_prompt: asks for JSON and lists every key", "JSON" in p and all(k in p for k in FIELDS))

PASS - build_prompt: no TODO left
PASS - build_prompt: the notice is in the prompt
PASS - build_prompt: the notice comes last
PASS - build_prompt: has a rule about missing details
PASS - build_prompt: asks for JSON and lists every key


## Run · Try your prompt

Run this cell. It sends your prompt and prints the raw reply, exactly as the model sent it.

In [7]:
result = safe_ask(build_prompt(notice_1), temperature=0)
print(result.get("response", "ERROR: " + result.get("error", "")))

{"title": "GDG AI workshop", "type": "event", "date": "2026-11-26", "time": "16:00", "venue": "Hall B", "registration_required": true}


Is it clean JSON? Does it start with `{`? Hold that thought, because Part 4 is about what to do when it doesn't.

---

# Part 3 · Show, don't just tell: examples in prompts (8 min)

## Read · Examples are the strongest instruction

Sometimes describing the format is not enough. Showing the model one or two finished examples is often stronger. Putting examples in a prompt is called **few-shot prompting**. A prompt with no examples is called **zero-shot**.

| | Zero-shot | Few-shot |
|---|---|---|
| What you give | Instructions only | Instructions + worked examples |
| Cost | Fewer tokens | More tokens (examples are paid for on every call) |
| Risk | Format may drift | The model may copy your examples too closely |

> Warning: **Never test your tool on the same notices you used as examples.** It will look better than it is. Keep examples and test cases separate. Week 3 is all about this.

## Run · Three worked examples

These are our examples. Notice they are **different** from each other: an event, a deadline, and an announcement with missing details.

In [8]:
EXAMPLES = [
    {
        "notice": "Reminder!! Chess club meets Wednesday 2nd Dec, 5pm, Room 12. Everyone welcome, no sign-up.",
        "json": {"title": "Chess club meeting", "type": "event", "date": "2026-12-02", "time": "17:00",
                 "venue": "Room 12", "registration_required": False},
    },
    {
        "notice": "Last day to pay school fees is 30th Nov. Pay at the Bursary before 3pm.",
        "json": {"title": "School fees payment", "type": "deadline", "date": "2026-11-30", "time": "15:00",
                 "venue": "Bursary", "registration_required": False},
    },
    {
        "notice": "Hostel inspection is coming. Keep your rooms tidy.",
        "json": {"title": "Hostel inspection", "type": "announcement", "date": None, "time": None,
                 "venue": None, "registration_required": False},
    },
]
print(len(EXAMPLES), "examples ready.")
print(json.dumps(EXAMPLES[2]["json"]))

3 examples ready.
{"title": "Hostel inspection", "type": "announcement", "date": null, "time": null, "venue": null, "registration_required": false}


Look at the third one: `null` is how JSON writes "nothing". Python shows it as `None`. The example teaches the model what to do when a detail is missing.

## Your turn · Exercise B: `build_prompt_with_examples`

Write a function that puts the examples **before** your instructions. It builds a block of text, then adds `build_prompt(notice)` after it.

Each example in the block should look like this:

```
Notice:
<the example notice>
JSON:
<the example json, as text>

```

One `TODO`: add one example to `block` inside the loop. Use `json.dumps(ex["json"])` to turn the example's dict into text.

<details><summary>Stuck? Click for the full answer</summary>

```python
block += f"Notice:\n{ex['notice']}\nJSON:\n{json.dumps(ex['json'])}\n\n"
```

</details>

In [9]:
def build_prompt_with_examples(notice, examples):
    block = "Here are examples of notices and the correct JSON:\n\n"
    for ex in examples:
        block += f"Notice:\n{ex['notice']}\nJSON:\n{json.dumps(ex['json'])}\n\n"
    return block + build_prompt(notice)

# --- check (don't edit below) ---
test_notice = "Quiz on Monday 7th Dec, 9am, LT 1."
p = build_prompt_with_examples(test_notice, EXAMPLES)
check("examples: every example notice is in the prompt", all(ex["notice"] in p for ex in EXAMPLES))
check("examples: every example JSON is in the prompt", all(json.dumps(ex["json"]) in p for ex in EXAMPLES))
check("examples: examples come before the real notice", p.find(EXAMPLES[0]["notice"]) < p.find(test_notice))
check("examples: the prompt still ends with the real notice", p.strip().endswith(test_notice))

PASS - examples: every example notice is in the prompt
PASS - examples: every example JSON is in the prompt
PASS - examples: examples come before the real notice
PASS - examples: the prompt still ends with the real notice


## Run · Zero-shot against few-shot

Same hard notice, two prompts. It uses slang and a relative date ("tmrw"), and the date is not written out.

In [10]:
tricky = "Free pizza at the Student Centre tmrw!! 1pm till it finishes lol"

zero = safe_ask(build_prompt(tricky), temperature=0)
time.sleep(1)
few = safe_ask(build_prompt_with_examples(tricky, EXAMPLES), temperature=0)

print("ZERO-SHOT:\n", zero.get("response", zero.get("error")).strip(), "\n")
print("FEW-SHOT:\n", few.get("response", few.get("error")).strip(), "\n")
print("Tokens sent: zero-shot", zero.get("prompt_tokens"), "| few-shot", few.get("prompt_tokens"))

ZERO-SHOT:
 {"title": "Free Pizza", "type": "event", "date": "2026-11-23", "time": "13:00", "venue": "Student Centre", "registration_required": false} 

FEW-SHOT:
 {"title": "Free pizza", "type": "event", "date": "2026-11-23", "time": "13:00", "venue": "Student Centre", "registration_required": false} 

Tokens sent: zero-shot 169 | few-shot 412


Compare the two answers, and compare the **tokens sent**. Examples improve results but cost more on every call. Both can be right. You choose.

---

# Part 4 · Asking for JSON (6 min)

## Read · Why JSON?

A program cannot use "The workshop is on Thursday at 4pm". It can use `{"date": "2026-11-26", "time": "16:00"}`. **JSON** is a text format that Python can turn into a dict. Last week you used `json.dumps` and `json.loads`. Now the model writes the JSON and you load it.

## Run · What the model actually sends back

Run this cell. The `repr` function shows the reply exactly, including hidden characters like new lines.

In [11]:
result = safe_ask(build_prompt(notice_1), temperature=0)
raw = result.get("response", "")
print(repr(raw))

'```json\n{\n  "title": "GDG AI workshop",\n  "type": "event",\n  "date": "2026-11-26",\n  "time": "16:00",\n  "venue": "Hall B",\n  "registration_required": true\n}\n```'


Many models wrap JSON in a code fence (three backticks and the word `json`), or add a polite sentence before or after it. A human can read past that. `json.loads` cannot.

## Run · Try to load it

Run this cell. If the reply is clean, it works. If it has a fence or extra words, you will see an error. Both are fine for learning.

In [12]:
try:
    data = json.loads(raw)
    print("Loaded fine:", data)
except json.JSONDecodeError as e:
    print("json.loads failed:", e)

json.loads failed: Expecting value: line 1 column 1 (char 0)


## Your turn · Exercise C: `parse_reply`

Write a function that turns the model's reply into Python data, even when there is extra text around the JSON.

The skeleton already tries `json.loads` on the whole reply. The `TODO` is the second attempt:

1. Find the first `{` in the text: `text.find("{")`
2. Find the last `}`: `text.rfind("}")`
3. Cut out that piece: `text[start:end + 1]`
4. Try `json.loads` on it. If it fails, return `None`.

<details><summary>Stuck? Click for the full answer</summary>

```python
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass
```

</details>

In [13]:
def parse_reply(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass
    return None

# --- check (don't edit below) ---
check("parse_reply: clean JSON", parse_reply('{"a": 1}') == {"a": 1})
check("parse_reply: JSON inside a code fence", parse_reply('```json\n{"a": 1}\n```') == {"a": 1})
check("parse_reply: JSON with chatter around it", parse_reply('Sure! Here you go: {"a": 1} Hope that helps!') == {"a": 1})
check("parse_reply: no JSON at all gives None", parse_reply("Sorry, I can't help with that.") is None)
check("parse_reply: broken JSON gives None", parse_reply('{"a": 1,}') is None)

PASS - parse_reply: clean JSON
PASS - parse_reply: JSON inside a code fence
PASS - parse_reply: JSON with chatter around it
PASS - parse_reply: no JSON at all gives None
PASS - parse_reply: broken JSON gives None


## Run · The JSON switch

Gemini has a setting that asks for JSON output. In our `ask` helper it is `json_mode=True`. Compare:

In [14]:
plain = safe_ask(build_prompt(notice_1), temperature=0)
time.sleep(1)
switch = safe_ask(build_prompt(notice_1), temperature=0, json_mode=True)

print("WITHOUT the switch:\n", repr(plain.get("response", plain.get("error"))), "\n")
print("WITH json_mode=True:\n", repr(switch.get("response", switch.get("error"))))

WITHOUT the switch:
 '{"title": "GDG AI workshop", "type": "event", "date": "2026-11-26", "time": "16:00", "venue": "Hall B", "registration_required": true}' 

WITH json_mode=True:
 '{\n  "title": "GDG AI workshop",\n  "type": "event",\n  "date": "2026-11-26",\n  "time": "16:00",\n  "venue": "Hall B",\n  "registration_required": true\n}'


> Warning: **The switch only helps with the shape of the text. It does not make the answer correct.** The model can still leave out a key, use `"yes"` instead of `true`, write the date in the wrong format, or invent a venue. Always validate. We keep `parse_reply` as well, because not every model has this switch.

---

# Part 5 · Validate what comes back (6 min)

## Read · "It's valid JSON" is not "it's correct"

Think of `parse_reply` as checking that the envelope opens. **Validation** checks what is inside. Six things can go wrong:

| What went wrong | Example | Caught by |
|---|---|---|
| Not JSON at all | "Sorry, I can't do that" | `parse_reply` returns `None` |
| A key is missing | no `"venue"` | `validate` |
| Wrong type | `"registration_required": "yes"` instead of `true` | `validate` |
| A value not on the allowed list | `"type": "party"` | `validate` |
| Wrong format | `"date": "26 Nov"` instead of `"2026-11-26"` | `validate` |
| **Valid, but false** | a venue that isn't in the notice | **Nothing here. Only testing catches this (Week 3)** |

## Run · Some bad replies to practise on

Run this cell. We made these replies up. Real ones will look similar.

In [15]:
DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")
TIME_PATTERN = re.compile(r"^([01]\d|2[0-3]):[0-5]\d$")

GOOD = {"title": "GDG AI workshop", "type": "event", "date": "2026-11-26", "time": "16:00",
        "venue": "Hall B", "registration_required": True}

BAD_REPLIES = {
    "not JSON": "I'm sorry, I couldn't find an event.",
    "missing key": '{"title": "GDG AI workshop", "type": "event", "date": "2026-11-26", "time": "16:00", "registration_required": true}',
    "wrong type": '{"title": "GDG AI workshop", "type": "event", "date": "2026-11-26", "time": "16:00", "venue": "Hall B", "registration_required": "yes"}',
    "bad type value": '{"title": "GDG AI workshop", "type": "party", "date": "2026-11-26", "time": "16:00", "venue": "Hall B", "registration_required": true}',
    "bad date format": '{"title": "GDG AI workshop", "type": "event", "date": "26th Nov", "time": "4pm", "venue": "Hall B", "registration_required": true}',
}

for name, reply in BAD_REPLIES.items():
    print(f"{name:<16} parse_reply gives:", parse_reply(reply))

not JSON         parse_reply gives: None
missing key      parse_reply gives: {'title': 'GDG AI workshop', 'type': 'event', 'date': '2026-11-26', 'time': '16:00', 'registration_required': True}
wrong type       parse_reply gives: {'title': 'GDG AI workshop', 'type': 'event', 'date': '2026-11-26', 'time': '16:00', 'venue': 'Hall B', 'registration_required': 'yes'}
bad type value   parse_reply gives: {'title': 'GDG AI workshop', 'type': 'party', 'date': '2026-11-26', 'time': '16:00', 'venue': 'Hall B', 'registration_required': True}
bad date format  parse_reply gives: {'title': 'GDG AI workshop', 'type': 'event', 'date': '26th Nov', 'time': '4pm', 'venue': 'Hall B', 'registration_required': True}


## Your turn · Exercise D: `validate`

Write a function that takes the parsed data and returns a **list of problems**. An empty list means the data is fine.

The skeleton checks that the data is a dict and that no key is missing. Four `TODO`s are left. In each one, add a problem message to the `problems` list when the check fails:

1. `type` must be one of `TYPES`
2. `date` must be `None`, or a text that matches `DATE_PATTERN` (use `DATE_PATTERN.match(...)`)
3. `time` must be `None`, or a text that matches `TIME_PATTERN`
4. `registration_required` must be `True` or `False` (use `isinstance(value, bool)`)

<details><summary>Stuck? Click for the full answer</summary>

```python
    if data.get("type") not in TYPES:
        problems.append("type must be one of " + ", ".join(TYPES))
    date = data.get("date")
    if date is not None and not (isinstance(date, str) and DATE_PATTERN.match(date)):
        problems.append("date must be YYYY-MM-DD or null")
    time_value = data.get("time")
    if time_value is not None and not (isinstance(time_value, str) and TIME_PATTERN.match(time_value)):
        problems.append("time must be HH:MM (24-hour) or null")
    if not isinstance(data.get("registration_required"), bool):
        problems.append("registration_required must be true or false")
```

</details>

In [16]:
def validate(data):
    problems = []
    if not isinstance(data, dict):
        return ["reply is not a JSON object"]
    for key in FIELDS:
        if key not in data:
            problems.append(f"missing key: {key}")
    if data.get("type") not in TYPES:
        problems.append("type must be one of " + ", ".join(TYPES))
    date = data.get("date")
    if date is not None and not (isinstance(date, str) and DATE_PATTERN.match(date)):
        problems.append("date must be YYYY-MM-DD or null")
    time_value = data.get("time")
    if time_value is not None and not (isinstance(time_value, str) and TIME_PATTERN.match(time_value)):
        problems.append("time must be HH:MM (24-hour) or null")
    if not isinstance(data.get("registration_required"), bool):
        problems.append("registration_required must be true or false")
    return problems

# --- check (don't edit below) ---
check("validate: a good reply has no problems", validate(GOOD) == [])
check("validate: spots a missing key", any("venue" in p for p in validate({k: v for k, v in GOOD.items() if k != "venue"})))
check("validate: rejects a type that is not allowed", validate({**GOOD, "type": "party"}) != [])
check("validate: rejects '26th Nov' as a date", validate({**GOOD, "date": "26th Nov"}) != [])
check("validate: rejects '4pm' as a time", validate({**GOOD, "time": "4pm"}) != [])
check("validate: rejects 'yes' for registration_required", validate({**GOOD, "registration_required": "yes"}) != [])
check("validate: null is allowed for date, time and venue", validate({**GOOD, "date": None, "time": None, "venue": None}) == [])
check("validate: None (no JSON) is a problem", validate(None) != [])

PASS - validate: a good reply has no problems
PASS - validate: spots a missing key
PASS - validate: rejects a type that is not allowed
PASS - validate: rejects '26th Nov' as a date
PASS - validate: rejects '4pm' as a time
PASS - validate: rejects 'yes' for registration_required
PASS - validate: null is allowed for date, time and venue
PASS - validate: None (no JSON) is a problem


## Run · Validate the bad replies

Now `parse_reply` and `validate` work together. Run this cell. Every bad reply should list at least one problem.

In [17]:
for name, reply in BAD_REPLIES.items():
    problems = validate(parse_reply(reply))
    print(f"{name:<16}", problems if problems else "no problems found")

not JSON         ['reply is not a JSON object']
missing key      ['missing key: venue']
wrong type       ['registration_required must be true or false']
bad type value   ['type must be one of event, deadline, announcement']
bad date format  ['date must be YYYY-MM-DD or null', 'time must be HH:MM (24-hour) or null']


## Run · Put it together: `extract`

This function chains everything: build the prompt, call the model, parse the reply, validate it. It returns a record that says what happened. **This is the core of your Component B.** Read it, then run it.

In [18]:
def extract(notice, examples=EXAMPLES):
    prompt = build_prompt_with_examples(notice, examples)
    result = safe_ask(prompt, temperature=0, json_mode=True)

    if "error" in result:
        return {"notice": notice, "raw": "", "data": None,
                "problems": ["the call failed: " + result["error"][:100]], "valid": False}

    raw = result["response"]
    data = parse_reply(raw)
    problems = validate(data)
    return {"notice": notice, "raw": raw, "data": data, "problems": problems, "valid": problems == []}

# Try it on the first notice
record = extract(notice_1)
print("valid:", record["valid"])
print("problems:", record["problems"])
print(json.dumps(record["data"], indent=2))

valid: True
problems: []
{
  "title": "GDG AI workshop",
  "type": "event",
  "date": "2026-11-26",
  "time": "16:00",
  "venue": "Hall B",
  "registration_required": true
}


Notice what `valid: True` does and does not mean: the **shape** is right. Is the *content* right? Is the date really 26 November 2026? Only you can check that, by reading. In Week 3 we teach the computer to check for us.

Run the scoreboard to see how Parts 1 to 5 went:

In [19]:
scoreboard()

Passed 22 of 22


---

# Part 6 · Lab: messy text to JSON (45 min, in pairs)

Work in pairs. One person types, the other reads and thinks, and you **swap every 15 minutes**. Ask your partner before you ask us.

## The notices

Ten made-up campus notices. They are deliberately messy: slang, missing details, relative dates, two dates in one notice. Notices 0 to 2 are for the lab. All ten are for the take-home.

In [20]:
NOTICES = [
    # 0
    notice_1,
    # 1
    "NOTICE: Deadline for submission of course registration forms is Friday 4th December 2026, 5:00 PM. Submit at the Registry. Late submissions attract a fine.",
    # 2
    "Career Talk: 'Breaking into Tech' with a guest speaker from a Lagos fintech. Auditorium, 2nd Dec, 2pm. Free entry, no need to register.",
    # 3
    "Library will be closed Friday 27th Nov for maintenance. Reopens Monday.",
    # 4
    "ATTN 300L CS students: Test 2 for CSC 301 holds this Tuesday, 24th Nov, 10am, LT 3. Bring your ID card.",
    # 5  (hard: two dates)
    "Hackathon registration closes Sunday 29 Nov at midnight!!! Teams of 3-4. Sign up at the Tech Hub desk or online. The hackathon itself is Sat 5 Dec, Lab 2, 9am.",
    # 6
    "Dear students, the Dean's lecture has been moved from Hall A to Hall C. Same time.",
    # 7
    tricky,
    # 8  (hard: two dates)
    "Scholarship application opens 1 Dec. Closes 15 January. Apply online with your CGPA and a recommendation letter.",
    # 9
    "WORKSHOP: Intro to Git - Saturday 28th Nov - 10:00am to 12:00pm - Lab 1 - registration compulsory via the form link",
]
print(len(NOTICES), "notices loaded.")
for i in (0, 4, 6):
    print(f"\n[{i}]", NOTICES[i])

10 notices loaded.

[0] HEY GUYS!!! GDG AI workshop this Thursday 26th Nov @ Hall B, 4pm sharp. Bring laptops.
Register at the link in bio. No registration = no entry! pls share

[4] ATTN 300L CS students: Test 2 for CSC 301 holds this Tuesday, 24th Nov, 10am, LT 3. Bring your ID card.

[6] Dear students, the Dean's lecture has been moved from Hall A to Hall C. Same time.


## Lab step 1 · Run the extractor on three notices (10 min)

Run it on notices 0, 1 and 2. Read each result. Is the JSON valid? Do the values look right?

In [21]:
lab_records = []
for i in (0, 1, 2):
    record = extract(NOTICES[i])
    lab_records.append(record)
    print(f"--- notice {i}: valid = {record['valid']}")
    print(NOTICES[i])
    print(json.dumps(record["data"], indent=2, ensure_ascii=False))
    if record["problems"]:
        print("PROBLEMS:", record["problems"])
    print()
    time.sleep(1)

--- notice 0: valid = True
HEY GUYS!!! GDG AI workshop this Thursday 26th Nov @ Hall B, 4pm sharp. Bring laptops.
Register at the link in bio. No registration = no entry! pls share
{
  "title": "GDG AI workshop",
  "type": "event",
  "date": "2026-11-26",
  "time": "16:00",
  "venue": "Hall B",
  "registration_required": true
}

--- notice 1: valid = True
NOTICE: Deadline for submission of course registration forms is Friday 4th December 2026, 5:00 PM. Submit at the Registry. Late submissions attract a fine.
{
  "title": "Course registration forms submission",
  "type": "deadline",
  "date": "2026-12-04",
  "time": "17:00",
  "venue": "Registry",
  "registration_required": false
}

--- notice 2: valid = True
Career Talk: 'Breaking into Tech' with a guest speaker from a Lagos fintech. Auditorium, 2nd Dec, 2pm. Free entry, no need to register.
{
  "title": "Career Talk: Breaking into Tech",
  "type": "event",
  "date": "2026-12-02",
  "time": "14:00",
  "venue": "Auditorium",
  "registra

## Lab step 2 · Compare with the answers we expected (15 min)

Below are the answers **we** think are right for notices 0, 1 and 2. The `compare` helper lists every field that is different. Run it.

A field that is different is not always wrong. Ask yourself each time: **who is right, the model or the expected answer?**

In [22]:
EXPECTED = {
    0: {"title": "GDG AI workshop", "type": "event", "date": "2026-11-26", "time": "16:00", "venue": "Hall B", "registration_required": True},
    1: {"title": "Course registration form submission", "type": "deadline", "date": "2026-12-04", "time": "17:00", "venue": "Registry", "registration_required": False},
    2: {"title": "Career Talk: Breaking into Tech", "type": "event", "date": "2026-12-02", "time": "14:00", "venue": "Auditorium", "registration_required": False},
}

def compare(got, expected):
    """List the fields where the model's answer is different from the expected answer."""
    if not isinstance(got, dict):
        return ["no data to compare"]
    return [f"{k}: model said {got.get(k)!r}, expected {v!r}" for k, v in expected.items() if got.get(k) != v]

for i, record in zip((0, 1, 2), lab_records):
    diffs = compare(record["data"], EXPECTED[i])
    print(f"notice {i}:", "matches" if not diffs else "")
    for d in diffs:
        print("   ", d)

notice 0: matches
notice 1: 
    title: model said 'Course registration forms submission', expected 'Course registration form submission'
notice 2: matches


Common differences: the **title** (the model words it differently, and that is fine) and the **time** or **date**. Titles are free text, so a different title can be correct. Dates and times are not free text, so a different one is a real error.

## Lab step 3 · Improve the prompt (15 min)

Pick **one** field that was wrong or missing in a result, and change **one thing** in `build_prompt` (a new rule, or a better example in `EXAMPLES`). Then rerun step 1 and step 2.

Change **one thing at a time**, or you won't know what helped. Write down what you changed and what happened:

In [23]:
LAB_NOTES = """
What was wrong: In the initial build_prompt draft, missing fields were instructed to be filled with "Unavailable" instead of null, and the notice text was hardcoded instead of using the {notice} variable dynamically.
What I changed: Updated build_prompt to pass {notice} at the end of the prompt string and instructed the model to return null whenever details are missing or ambiguous.
What happened: All build_prompt checks passed, and the extractor cleanly outputted valid JSON with null values for unspecified dates, times, and venues instead of invalid placeholder strings.
"""

## Lab step 4 · Try the hard ones (5 min)

Notices 5 and 8 each have **two dates**. Run them and look at what the model does. Which date does it pick? Is there a "right" answer? What would you want your tool to do?

In [24]:
for i in (5, 8):
    record = extract(NOTICES[i])
    print(f"--- notice {i}: valid = {record['valid']}")
    print(NOTICES[i])
    print(json.dumps(record["data"], indent=2, ensure_ascii=False))
    print("PROBLEMS:", record["problems"])
    print()
    time.sleep(1)

--- notice 5: valid = False
Hackathon registration closes Sunday 29 Nov at midnight!!! Teams of 3-4. Sign up at the Tech Hub desk or online. The hackathon itself is Sat 5 Dec, Lab 2, 9am.
[
  {
    "title": "Hackathon registration",
    "type": "deadline",
    "date": "2026-11-29",
    "time": "00:00",
    "venue": "Tech Hub desk or online",
    "registration_required": true
  },
  {
    "title": "Hackathon",
    "type": "event",
    "date": "2026-12-05",
    "time": "09:00",
    "venue": "Lab 2",
    "registration_required": true
  }
]
PROBLEMS: ['reply is not a JSON object']

--- notice 8: valid = True
Scholarship application opens 1 Dec. Closes 15 January. Apply online with your CGPA and a recommendation letter.
{
  "title": "Scholarship application",
  "type": "deadline",
  "date": "2027-01-15",
  "time": null,
  "venue": null,
  "registration_required": true
}
PROBLEMS: []



Our schema has **one** date, so a notice with two dates does not fit it. The tool is not "wrong". The schema is too small for the data. Real projects hit this all the time. Write down how you would change the schema. You don't have to build it.

---

# Part 7 · Take-home: Project 1, Component B

**Component B = an extractor that runs over 10 messy inputs and checks that each result is valid.**

Time: about 3 to 4 hours. You reuse everything from today.

| Week | What you build |
|---|---|
| 1 | **Component A:** `safe_ask` and your 5 prompts (done) |
| 2 (now) | **Component B:** the extractor, run over 10 notices, with validity checks |
| 3 | **Component C:** a test runner that prints a pass rate, plus one measured improvement |
| 4 | **Project 1 session:** put A, B and C together. Demos and live fixes |

## Step 1 · Finish your extractor

Your `build_prompt`, `build_prompt_with_examples`, `parse_reply`, `validate` and `extract` are the extractor. Make sure all the exercise checks above are PASS (the scoreboard tells you).

## Step 2 · Run it on all 10 notices

Fill in the `TODO`s. For each notice, call `extract` and add the record to `records`. Then save the list to `extracted.json` with `save_json`.

In [25]:
records = []

for notice in NOTICES:
    records.append(extract(notice))
    time.sleep(1)

save_json(records, "extracted.json")

## Step 3 · Count the valid ones

Count how many of the 10 records are valid, and print the notices that were not. `record["valid"]` is `True` or `False`. Fill in the `TODO`.

In [26]:
valid_count = sum(1 for r in records if r["valid"])

print(f"{valid_count} of {len(records)} replies were valid.")
for i, record in enumerate(records):
    if not record["valid"]:
        print(f"  notice {i} failed:", record["problems"])

10 of 10 replies were valid.


## Step 4 · Read three results by eye

Valid does not mean correct. Pick **three** valid records and read them against their notices. Write down anything that is valid but wrong, for example a wrong date or a made-up venue. At least a few sentences.

In [27]:
EYE_CHECK = """
Notice number: 3
What I found: The notice states the library closes Friday 27th Nov and reopens Monday, without giving a specific time or registration requirement. The JSON schema passed validation with date "2026-11-27", time null, venue "Library", and registration_required false, which accurately captures the closure date without inventing a time.

Notice number: 5
What I found: This notice contains two different dates—the registration deadline on Sunday 29 Nov at midnight and the actual hackathon on Saturday 5 Dec at 9am in Lab 2. Because the schema only supports a single date field, the model had to pick one event date while still passing schema validation.

Notice number: 6
What I found: The announcement mentions that the Dean's lecture moved from Hall A to Hall C at the "Same time" with no date or clock time provided. The extractor returned valid JSON with venue "Hall C" and properly set date and time to null rather than hallucinating values.
"""

## Step 5 · Check your work

In [28]:
saved = []
file_text = ""
if os.path.exists("extracted.json"):
    file_text = open("extracted.json", encoding="utf-8").read()
    saved = json.loads(file_text)

keys_ok = len(saved) > 0
for r in saved:
    if not {"notice", "raw", "data", "problems", "valid"} <= set(r):
        keys_ok = False

check("extracted.json exists", os.path.exists("extracted.json"))
check("it has 10 records", len(saved) == 10)
check("every record has notice, raw, data, problems and valid", keys_ok)
check("valid_count is right", valid_count == sum(1 for r in saved if r.get("valid")) and len(saved) > 0)
check("no API key inside extracted.json", API_KEY == "" or API_KEY not in file_text)
check("you read three results by eye", len(EYE_CHECK.strip()) >= 100)
check("you wrote lab notes", len(LAB_NOTES.strip()) >= 80 and LAB_NOTES.count(":") >= 3)
scoreboard()

PASS - extracted.json exists
PASS - it has 10 records
PASS - every record has notice, raw, data, problems and valid
PASS - valid_count is right
PASS - no API key inside extracted.json
PASS - you read three results by eye
PASS - you wrote lab notes
Passed 29 of 29


Open `extracted.json` in VS Code and read it. Real engineers look at their data.

## Step 6 · What broke?

Write **one paragraph**: the biggest thing that went wrong or surprised you this week, and how you dealt with it. This is practice for the 1-page "what broke" write-up in Week 4.

In [29]:
WHAT_BROKE = """
The biggest challenge this week was handling ambiguous inputs that technically passed schema validation but exposed the limits of a single-record JSON design. In particular, notices containing two separate dates—such as a registration deadline followed by the main event date, or an application opening and closing window—forced the model to choose only one date to fit our single date key. Additionally, forgetting to pass the dynamic notice parameter at the end of the prompt template initially caused the model to extract from a static string instead of the actual input. Adding strict null rules, isolating the JSON extraction block in parse_reply, and validating types in Python resolved the formatting failures, while manually inspecting extracted.json showed why content-level testing is essential next week.
"""

check("you wrote your 'what broke' paragraph", len(WHAT_BROKE.strip()) >= 80)

PASS - you wrote your 'what broke' paragraph


## Stretch (optional): retry when the output is invalid

If a record is not valid, call the model **again**, this time telling it what was wrong. Try up to 2 retries. Write `extract_with_retry(notice)`. Then count how many more notices become valid.

Hint: build a second prompt like `build_prompt_with_examples(notice, EXAMPLES) + "\n\nYour last reply had these problems: ..." + str(problems)`. Then call `parse_reply` and `validate` again.

## Step 7 · Submit

1. Save the notebook (`Ctrl + S`).
2. Open your `gdg-ai-track` repo on github.com, click **Add file → Upload files**, and upload this notebook and `extracted.json`. Write a commit message such as "Add Week 2: extractor", then click **Commit changes**.
3. Send your track lead the repo link and your "what broke" paragraph before the next session.

> Warning: Before uploading, scroll through the notebook's outputs. If your API key ever appeared on screen, delete that output and make a new key. Never upload your `.env` file.

---

# Glossary

| Term | Plain meaning |
|---|---|
| Prompt | The text you send to the model: instructions plus input |
| System instruction | A prompt part that sets the role and tone |
| Zero-shot | A prompt with instructions only, no examples |
| Few-shot | A prompt that includes worked examples |
| JSON | A text format for data that programs share |
| Parse | Turn text (like JSON) into Python data |
| Schema | The agreed shape of your data: which keys, which types, which values are allowed |
| Validate | Check that data matches the schema |
| `null` / `None` | "No value". JSON writes `null`, Python writes `None` |
| Hallucination | A confident, believable, false statement |
| Retry | Try a failed step again |

# Next week

**Week 3: Testing your AI tool.** Today you checked that the *shape* of the output is right. Next week you test that the *content* is right, with a list of test cases and a pass rate, and you make one measured improvement. That is **Component C**.

Bring your `extracted.json`. You will build your test cases from the notices that gave you trouble.

---
### Track-lead notes (delete this cell before sharing the notebook)

- **Curriculum source:** Week 2 in the v5 workbook: prompts as instructions, examples in prompts, JSON output, validating what comes back. Lab: messy text to JSON. Take-home: Component B, an extractor over 10 messy inputs with a JSON validity check. Stretch: automatic retry when the output is invalid.
- **Run the notebook yourself first**, with your key and your internet. The exercise checks (A to D) run offline and were tested. The live cells (everything that calls the model) were **not** run against Gemini by the author. In particular, `json_mode=True` (`response_mime_type="application/json"`) and the model name `gemini-3.8-flash` are unverified.
- **Offline fallback:** exercises A to D, `compare`, and Parts 5 to 6 reading work without internet. Live cells need it. Run the whole notebook at home and save the outputs first, then present from the saved copy if the room's wifi fails.
- **Likely stuck points:** (1) skipped cells, then `NameError`; (2) forgetting to restart the kernel after fixing `.env`; (3) exercise D: `bool` vs text `"true"`; (4) `429` when the whole room calls at once, so stagger the lab step 1; (5) `404` if the model was renamed.
- **Expected answers are opinions.** Notice 4 (a test) could be `event` or `announcement`. Notices 5 and 8 have two dates and do not fit the schema on purpose. Use them to start the "what does correct mean?" discussion that leads into Week 3.
- **Project name and numbers:** this notebook follows the v5 curriculum (Project 1: Campus Notice Extractor, Week 4, solo). If your Week 1 notebook said "Document Extractor" or a 30-case golden set, align the two before class.
- **Timing (estimates, not tested in a room):** concept Parts 1 to 5 is tight at 35 minutes. If you run long, skip the zero-shot vs few-shot comparison and the JSON switch demo, and set them as reading.